# 3. Geographic Health Equity Analysis
## Regional Healthcare Access and Disparity Intelligence Pipeline

Strategy: LIMIT 1000 on BigQuery reads -> Local CSV -> Pandas -> Spark

## Available Tables:
- **OMOP**: 24 tables
- **Medicare**: 6 tables
- **Dual Enrollment**: 1 table (SDOH)
- **CMS Codes**: 3 tables

## Pipeline Architecture:
- **Bronze**: 11 tables (raw geographic & clinical data)
- **Silver**: 6 intermediate layers (regional aggregations)
- **Gold**: 15 vertical layers -> 4 final metrics

## Final Metrics:
1. **Health Equity Index by Region** - Comprehensive access + outcome integration
2. **Care Desert Score** - Medical service availability index
3. **Social Determinants Impact** - SDOH influence measurement
4. **Geographic Disparity Magnitude** - Regional inequality quantification

In [1]:
!pip install google-cloud-bigquery
!pip install pandas
!pip install networkx


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from google.cloud import bigquery
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import *
import os
from datetime import datetime
import networkx as nx
import json
import requests 

In [3]:
print("Initializing Spark...")
spark = (
    SparkSession.builder 
    .appName("CMS_GeographicHealthEquity") 
    .master("local[*]") 
    
    # OpenLineage Configuration
    .config("spark.jars.packages", "io.openlineage:openlineage-spark_2.12:1.18.0")
    .config("spark.extraListeners", "io.openlineage.spark.agent.OpenLineageSparkListener")
    .config("spark.openlineage.transport.type", "http")
    .config("spark.openlineage.transport.url", "http://localhost:4601")
    .config("spark.openlineage.namespace", "geographic_equity")

    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.sql.shuffle.partitions", "8") 
    .config("spark.driver.maxResultSize", "2g") 
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"✅ OpenLineage → http://localhost:4601")
print(f"✅ Namespace: geographic_equity")

Initializing Spark...
:: loading settings :: url = jar:file:/Users/dhananjaysharma/Desktop/provenance-local/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/dhananjaysharma/.ivy2/cache
The jars for the packages stored in: /Users/dhananjaysharma/.ivy2/jars
io.openlineage#openlineage-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7c3f5c6b-a691-43f5-a632-01cf983d7dcf;1.0
	confs: [default]
	found io.openlineage#openlineage-spark_2.12;1.18.0 in central
:: resolution report :: resolve 40ms :: artifacts dl 1ms
	:: modules in use:
	io.openlineage#openlineage-spark_2.12;1.18.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submi

Spark version: 3.5.1
Spark UI: http://mac:4040
✅ OpenLineage → http://localhost:4601
✅ Namespace: geographic_equity


In [4]:
# Check Marquez Connection
print("\nChecking Marquez connection...")
try:
    response = requests.get("http://localhost:4601/api/v1/namespaces", timeout=2)
    if response.status_code == 200:
        print("✅ Marquez is running at http://localhost:4601")
        print("✅ Web UI: http://localhost:3601")
    else:
        print("⚠️ Marquez responded but might have issues")
except Exception as e:
    print("❌ Cannot connect to Marquez!")
    print("   Please run: docker-compose -f docker-compose-enhanced.yml up -d")


Checking Marquez connection...
✅ Marquez is running at http://localhost:4601
✅ Web UI: http://localhost:3601


In [5]:
PROJECT_ID = "opportune-ruler-447319-b3"
LIMIT = 1000
LOCAL_DATA_DIR = "./3_data"

# os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

DATASETS = {
    "omop": "bigquery-public-data.cms_synthetic_patient_data_omop",
    "medicare": "bigquery-public-data.cms_medicare",
    "dual": "bigquery-public-data.sdoh_cms_dual_eligible_enrollment",
    "hcpcs": "bigquery-public-data.cms_codes"
}

print(f"Config:")
print(f"  Project: {PROJECT_ID}")
print(f"  Limit: {LIMIT} rows per table")
print(f"  Data dir: {LOCAL_DATA_DIR}")

Config:
  Project: opportune-ruler-447319-b3
  Limit: 1000 rows per table
  Data dir: ./3_data


# STEP 1: Download from BigQuery (LIMIT 1000)

In [6]:
def download_table(client, dataset_key, table_name, limit=1000):
    """Download table from BigQuery to local CSV"""
    try:
        source = DATASETS[dataset_key]
        query = f"SELECT * FROM `{source}.{table_name}` LIMIT {limit}"
        df = client.query(query).to_dataframe()
        
        output_file = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        df.to_csv(output_file, index=False)
        print(f"  ✓ {dataset_key}.{table_name}: {len(df)} rows -> {output_file}")
        return True
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: {str(e)}")
        return False

In [7]:
print("\n" + "="*60)
print("DOWNLOADING TABLES")
print("="*60)

client = bigquery.Client(project=PROJECT_ID)

tables_to_download = [
    ("omop", "person"),
    ("omop", "location"),
    ("omop", "care_site"),
    ("omop", "provider"),
    ("omop", "condition_occurrence"),
    ("omop", "procedure_occurrence"),
    ("omop", "drug_exposure"),
    ("omop", "observation_period"),
    ("dual", "dual_eligible_enrollment_by_county_and_program"),
    ("medicare", "hospital_general_info"),
    ("medicare", "inpatient_charges_2011")
]

for dataset_key, table_name in tables_to_download:
    download_table(client, dataset_key, table_name, LIMIT)

print("\n✓ Download complete")


DOWNLOADING TABLES
  ✗ omop.person: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/opportune-ruler-447319-b3/jobs?prettyPrint=false: Access Denied: Project opportune-ruler-447319-b3: User does not have bigquery.jobs.create permission in project opportune-ruler-447319-b3.

Location: None
Job ID: e21855d5-5d9d-4054-814f-1f2d579b1b2d

  ✗ omop.location: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/opportune-ruler-447319-b3/jobs?prettyPrint=false: Access Denied: Project opportune-ruler-447319-b3: User does not have bigquery.jobs.create permission in project opportune-ruler-447319-b3.

Location: None
Job ID: ec1bd190-e82e-4934-8db3-d0126a9d1a96

  ✗ omop.care_site: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/opportune-ruler-447319-b3/jobs?prettyPrint=false: Access Denied: Project opportune-ruler-447319-b3: User does not have bigquery.jobs.create permission in project opportune-ruler-447319-b3.

Location: None
Job ID: bf3ddc3f-e0c9-4caf-8843

# STEP 2: Load CSV -> Pandas -> Spark (Bronze Layer)

In [8]:
def load_csv_to_spark_with_lineage(dataset_key, table_name, layer="bronze"):
    """Load CSV file into Spark - NO metadata columns, NO count()"""
    try:
        csv_path = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        
        if not os.path.exists(csv_path):
            print(f"  ✗ {dataset_key}.{table_name}: CSV not found at {csv_path}")
            return None
        
        # Read CSV - NO metadata, NO count()
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(csv_path)
        
        print(f"  ✓ {dataset_key}.{table_name}: loaded into {layer} layer")
        return df
        
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: Error - {str(e)}")
        return None


def add_layer_metadata(df, layer, source_tables, target_table=None):
    """DEPRECATED - Don't use metadata columns"""
    return df  # Just return DataFrame as-is

In [ ]:
print("\n" + "="*60)
print("LOADING AND WRITING BRONZE LAYER")
print("="*60)

os.makedirs("./output/bronze", exist_ok=True)

def load_and_write_bronze(dataset_key, table_name):
    """Load CSV and write immediately to capture lineage"""
    df = load_csv_to_spark_with_lineage(dataset_key, table_name, "bronze")
    if df is not None:
        output_path = f"./output/bronze/bronze_{dataset_key}_{table_name}"
        df.write.mode("overwrite").parquet(output_path)
        print(f"    → Written to bronze")
        # Read back for use in Silver
        return spark.read.parquet(output_path)
    return None

# Load and write all Bronze tables
bronze_person = load_and_write_bronze("omop", "person")
bronze_location = load_and_write_bronze("omop", "location")
bronze_care_site = load_and_write_bronze("omop", "care_site")
bronze_provider = load_and_write_bronze("omop", "provider")
bronze_condition = load_and_write_bronze("omop", "condition_occurrence")
bronze_procedure = load_and_write_bronze("omop", "procedure_occurrence")
bronze_drug = load_and_write_bronze("omop", "drug_exposure")
bronze_obs_period = load_and_write_bronze("omop", "observation_period")
bronze_dual_enroll = load_and_write_bronze("dual", "dual_eligible_enrollment_by_county_and_program")
bronze_hospital_info = load_and_write_bronze("medicare", "hospital_general_info")
bronze_inpatient = load_and_write_bronze("medicare", "inpatient_charges_2011")

print(f"\n✓ Bronze layer loaded and written (11 tables)")


LOADING AND WRITING BRONZE LAYER


25/11/23 20:06:34 WARN RddPathUtils: Unknown RDD class SQLExecutionRDD[7] at csv at NativeMethodAccessorImpl.java:0
25/11/23 20:06:34 WARN RddPathUtils: Unknown RDD class SQLExecutionRDD[7] at csv at NativeMethodAccessorImpl.java:0
25/11/23 20:06:34 WARN RddPathUtils: Unknown RDD class SQLExecutionRDD[7] at csv at NativeMethodAccessorImpl.java:0
25/11/23 20:06:34 WARN RddPathUtils: Unknown RDD class SQLExecutionRDD[7] at csv at NativeMethodAccessorImpl.java:0


  ✓ omop.person: loaded into bronze layer
    → Written to bronze
  ✓ omop.location: loaded into bronze layer


25/11/23 20:06:34 ERROR ContextFactory: Query execution is null: can't emit event for executionId 2
25/11/23 20:06:34 ERROR ContextFactory: Query execution is null: can't emit event for executionId 2


    → Written to bronze
  ✓ omop.care_site: loaded into bronze layer
    → Written to bronze


...

  ✓ omop.condition_occurrence: loaded into bronze layer
    → Written to bronze
  ✓ omop.procedure_occurrence: loaded into bronze layer
    → Written to bronze
  ✓ omop.drug_exposure: loaded into bronze layer


25/11/23 20:06:35 ERROR ContextFactory: Query execution is null: can't emit event for executionId 10
25/11/23 20:06:35 ERROR ContextFactory: Query execution is null: can't emit event for executionId 10
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 11
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 11
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 12
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 12


    → Written to bronze
  ✓ omop.observation_period: loaded into bronze layer
    → Written to bronze
  ✓ dual.dual_eligible_enrollment_by_county_and_program: loaded into bronze layer
    → Written to bronze
  ✓ medicare.hospital_general_info: loaded into bronze layer


25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 14
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 14
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 15
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 15
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 16
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 16
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 17
25/11/23 20:06:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 17
25/11/23 20:06:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/

    → Written to bronze
  ✓ medicare.inpatient_charges_2011: loaded into bronze layer
    → Written to bronze

✓ Bronze layer loaded and written (11 tables)


# STEP 3: Build SILVER Layer (6 Regional Aggregations)

In [10]:
print("\n" + "="*60)
print("SILVER 1/6: Regional Demographics Profile")
print("="*60)

os.makedirs("./output/silver", exist_ok=True)

silver_regional_demographics = bronze_person \
    .join(bronze_location, bronze_person.location_id == bronze_location.location_id, "left") \
    .groupBy("state", "zip", "county") \
    .agg(
        F.count("*").alias("population_count"),
        F.avg("year_of_birth").alias("avg_birth_year"),
        F.countDistinct("gender_concept_id").alias("gender_diversity"),
        F.countDistinct("race_concept_id").alias("race_diversity"),
        F.countDistinct("ethnicity_concept_id").alias("ethnicity_diversity")
    ) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_regional_demographics.count()}")

silver_regional_demographics = add_layer_metadata(
    df=silver_regional_demographics,
    layer="silver",
    source_tables=["bronze_person"],
    target_table="silver_regional_demographics"
)

# Write immediately!
silver_regional_demographics.write.mode("overwrite").parquet("./output/silver/silver_regional_demographics")
print(f"✓ Regional demographics written")

# Read back
silver_regional_demographics = spark.read.parquet("./output/silver/silver_regional_demographics")
print(f"  Regions: {silver_regional_demographics.count()}")

silver_regional_demographics.show(5, truncate=False)


SILVER 1/6: Regional Demographics Profile


25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 20
25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 20
25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 21
25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 21


  Regions: 191
✓ Regional demographics written
  Regions: 191
+-----+----+------+----------------+------------------+----------------+--------------+-------------------+--------------------------+
|state|zip |county|population_count|avg_birth_year    |gender_diversity|race_diversity|ethnicity_diversity|processed_at              |
+-----+----+------+----------------+------------------+----------------+--------------+-------------------+--------------------------+
|CT   |NULL|7010  |3               |1932.0            |1               |1             |1                  |2025-11-23 20:06:37.301615|
|GA   |NULL|11260 |1               |1974.0            |1               |1             |1                  |2025-11-23 20:06:37.301615|
|ID   |NULL|13000 |3               |1976.3333333333333|1               |1             |1                  |2025-11-23 20:06:37.301615|
|FL   |NULL|10100 |1               |1919.0            |1               |1             |1                  |2025-11-23 20:06:37.3

In [11]:
print("\n" + "="*60)
print("SILVER 2/6: Provider Density by Region")
print("="*60)

provider_with_location = bronze_provider \
    .join(bronze_care_site, "care_site_id", "left") \
    .join(bronze_location, bronze_care_site.location_id == bronze_location.location_id, "left")

silver_provider_density = provider_with_location \
    .groupBy("state", "zip", "county") \
    .agg(
        F.count("*").alias("total_providers"),
        F.countDistinct("specialty_concept_id").alias("specialty_diversity"),
        F.countDistinct("care_site_id").alias("care_sites_count"),
        F.avg("year_of_birth").alias("avg_provider_age")
    ) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_provider_density.count()}")

silver_provider_density = add_layer_metadata(
    df=silver_provider_density,  
    layer="silver",
    source_tables=["bronze_provider", "bronze_care_site", "bronze_location"],  
    target_table="silver_provider_density"
)

# Write immediately!
silver_provider_density.write.mode("overwrite").parquet("./output/silver/silver_provider_density")
print(f"✓ Provider Density written")

# Read back
silver_provider_density = spark.read.parquet("./output/silver/silver_provider_density")
print(f"  Provider Density: {silver_provider_density.count()}")

silver_provider_density.show(5, truncate=False)


SILVER 2/6: Provider Density by Region
  Regions: 1


25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 24
25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 24
25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 24
25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 25
25/11/23 20:06:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 25


✓ Provider Density written
  Provider Density: 1
+-----+----+------+---------------+-------------------+----------------+----------------+--------------------------+
|state|zip |county|total_providers|specialty_diversity|care_sites_count|avg_provider_age|processed_at              |
+-----+----+------+---------------+-------------------+----------------+----------------+--------------------------+
|NULL |NULL|NULL  |1000           |0                  |162             |NULL            |2025-11-23 20:06:37.830551|
+-----+----+------+---------------+-------------------+----------------+----------------+--------------------------+



In [12]:
print("\n" + "="*60)
print("SILVER 3/6: Regional Service Utilization")
print("="*60)

person_location = bronze_person \
    .join(bronze_location, "location_id", "left") \
    .select("person_id", "state", "zip", "county")

condition_regional = bronze_condition \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county") \
    .agg(F.count("*").alias("condition_events"))

procedure_regional = bronze_procedure \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county") \
    .agg(F.count("*").alias("procedure_events"))

drug_regional = bronze_drug \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county") \
    .agg(F.count("*").alias("drug_events"))

silver_service_utilization = condition_regional \
    .join(procedure_regional, ["state", "zip", "county"], "outer") \
    .join(drug_regional, ["state", "zip", "county"], "outer") \
    .fillna(0) \
    .withColumn("total_events", 
        F.col("condition_events") + F.col("procedure_events") + F.col("drug_events")) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_service_utilization.count()}")
silver_service_utilization.show(5, truncate=False)

silver_service_utilization = add_layer_metadata(
    df=silver_service_utilization,
    layer="silver",
    source_tables=["bronze_condition", "bronze_procedure", "bronze_drug", "bronze_person", "bronze_location"],
    target_table="silver_service_utilization"
)

# Write immediately!
silver_service_utilization.write.mode("overwrite").parquet("./output/silver/silver_service_utilization")
print(f"✓ Service written")

# Read back
silver_service_utilization = spark.read.parquet("./output/silver/silver_service_utilization")
print(f"  Service: {silver_service_utilization.count()}")


SILVER 3/6: Regional Service Utilization


25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 28
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 28
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 28
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 29
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 29
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 30
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 30
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 30
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 30
25/11/23 20:06:38 ERROR ContextFactory: Query execution is null: can't emit event for execu

  Regions: 3
+-----+----+------+----------------+----------------+-----------+------------+--------------------------+
|state|zip |county|condition_events|procedure_events|drug_events|total_events|processed_at              |
+-----+----+------+----------------+----------------+-----------+------------+--------------------------+
|NULL |NULL|0     |1000            |0               |0          |1000        |2025-11-23 20:06:38.759072|
|NULL |NULL|0     |0               |1000            |0          |1000        |2025-11-23 20:06:38.759072|
|NULL |NULL|0     |0               |0               |1000       |1000        |2025-11-23 20:06:38.759072|
+-----+----+------+----------------+----------------+-----------+------------+--------------------------+

✓ Service written
  Service: 3


In [13]:
print("\n" + "="*60)
print("SILVER 4/6: Regional Health Outcomes")
print("="*60)

chronic_conditions = bronze_condition \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county", "person_id") \
    .agg(F.countDistinct("condition_concept_id").alias("condition_count"))

silver_health_outcomes = chronic_conditions \
    .groupBy("state", "zip", "county") \
    .agg(
        F.avg("condition_count").alias("avg_conditions_per_patient"),
        F.max("condition_count").alias("max_conditions_per_patient"),
        F.count(F.when(F.col("condition_count") >= 3, 1)).alias("complex_patients_count")
    ) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_health_outcomes.count()}")
silver_health_outcomes.show(5, truncate=False)

silver_health_outcomes = add_layer_metadata(
    df=silver_health_outcomes,
    layer="silver",
    source_tables=["bronze_condition", "bronze_person", "bronze_location"],
    target_table="silver_health_outcomes"
)

# Write immediately!
silver_health_outcomes.write.mode("overwrite").parquet("./output/silver/silver_health_outcomes")
print(f"✓ Health Outcomes written")

# Read back
silver_health_outcomes = spark.read.parquet("./output/silver/silver_health_outcomes")
print(f"  Health Outcomes: {silver_health_outcomes.count()}")


SILVER 4/6: Regional Health Outcomes
  Regions: 1
+-----+----+------+--------------------------+--------------------------+----------------------+--------------------------+
|state|zip |county|avg_conditions_per_patient|max_conditions_per_patient|complex_patients_count|processed_at              |
+-----+----+------+--------------------------+--------------------------+----------------------+--------------------------+
|NULL |NULL|NULL  |1.0                       |1                         |0                     |2025-11-23 20:06:39.464652|
+-----+----+------+--------------------------+--------------------------+----------------------+--------------------------+

✓ Health Outcomes written
  Health Outcomes: 1


In [14]:
print("\n" + "="*60)
print("SILVER 5/6: SDOH (Social Determinants of Health) Indicators")
print("="*60)

if bronze_dual_enroll is not None:
    silver_sdoh_indicators = bronze_dual_enroll \
        .groupBy("State_Abbr", "County_Name") \
        .agg(
            F.sum("Public_Total").alias("total_dual_eligible"),
            F.sum("QMB_plus_Full").alias("qmb_plus_full_count"),
            F.sum("QMB_Only").alias("qmb_only_count"),
            F.sum("SLMB_only").alias("slmb_only_count"),
            F.sum("SLMB_plus_Full").alias("slmb_plus_full_count"),
            F.sum("QDWI").alias("qdwi_count"),
            F.sum("QI").alias("qi_count"),
            F.sum("Other_full").alias("other_dual_count")
        ) \
        .withColumn("dual_eligible_ratio",
            F.col("qmb_plus_full_count") / (F.col("total_dual_eligible") + 1)) \
        .withColumn("processed_at", F.current_timestamp())
    
    print(f"  Regions: {silver_sdoh_indicators.count()}")
    silver_sdoh_indicators.show(5, truncate=False)

    silver_sdoh_indicators = add_layer_metadata(
        df=silver_sdoh_indicators,
        layer="silver",
        source_tables=["bronze_dual_enroll"],
        target_table="silver_sdoh_indicators"
    )

    # Write immediately!
    silver_sdoh_indicators.write.mode("overwrite").parquet("./output/silver/silver_sdoh_indicators")
    print(f"✓ SDOH Indicators written")

    # Read back
    silver_sdoh_indicators = spark.read.parquet("./output/silver/silver_sdoh_indicators")
    print(f"  SDOH Indicators: {silver_sdoh_indicators.count()}")
else:
    print("  ✗ Dual enrollment data not available")
    silver_sdoh_indicators = None


SILVER 5/6: SDOH (Social Determinants of Health) Indicators
  Regions: 125
+----------+-----------+-------------------+-------------------+--------------+---------------+--------------------+----------+--------+----------------+-------------------+--------------------------+
|State_Abbr|County_Name|total_dual_eligible|qmb_plus_full_count|qmb_only_count|slmb_only_count|slmb_plus_full_count|qdwi_count|qi_count|other_dual_count|dual_eligible_ratio|processed_at              |
+----------+-----------+-------------------+-------------------+--------------+---------------+--------------------+----------+--------+----------------+-------------------+--------------------------+
|AL        |BULLOCK    |4172               |1500               |1350          |599            |NULL                |0         |359     |364             |0.35945363048166784|2025-11-23 20:06:39.941123|
|AL        |CHOCTAW    |7299               |2798               |2349          |981            |214                 |0   

In [15]:
print("\n" + "="*60)
print("SILVER 6/6: Facility Access and Quality")
print("="*60)

if bronze_hospital_info is not None:
    silver_facility_access = bronze_hospital_info \
        .groupBy("state", "zip_code", "county_name") \
        .agg(
            F.count("*").alias("hospital_count"),
            F.countDistinct("hospital_type").alias("hospital_type_diversity"),
            F.countDistinct("hospital_ownership").alias("ownership_diversity"),
            F.avg("hospital_overall_rating").alias("avg_hospital_rating")
        ) \
        .withColumn("processed_at", F.current_timestamp())
    
    print(f"  Regions: {silver_facility_access.count()}")
    silver_facility_access.show(5, truncate=False)

    silver_facility_access = add_layer_metadata(
        df=silver_facility_access,
        layer="silver",
        source_tables=["bronze_hospital_info"],
        target_table="silver_facility_access"
    )

    # Write immediately!
    silver_facility_access.write.mode("overwrite").parquet("./output/silver/silver_facility_access")
    print(f"✓ Facility Access written")

    # Read back
    silver_facility_access = spark.read.parquet("./output/silver/silver_facility_access")
    print(f"  Facility Access: {silver_facility_access.count()}")
else:
    print("  ✗ Hospital info data not available")
    silver_facility_access = None

print("\n✓ Silver Layer complete (6 tables)")


SILVER 6/6: Facility Access and Quality
  Regions: 984
+-----+--------+-----------+--------------+-----------------------+-------------------+-------------------+--------------------------+
|state|zip_code|county_name|hospital_count|hospital_type_diversity|ownership_diversity|avg_hospital_rating|processed_at              |
+-----+--------+-----------+--------------+-----------------------+-------------------+-------------------+--------------------------+
|OH   |43110   |Franklin   |1             |1                      |1                  |NULL               |2025-11-23 20:06:40.273849|
|NE   |68122   |Douglas    |1             |1                      |1                  |NULL               |2025-11-23 20:06:40.273849|
|CA   |92225   |Riverside  |1             |1                      |1                  |NULL               |2025-11-23 20:06:40.273849|
|KS   |67950   |Morton     |1             |1                      |1                  |NULL               |2025-11-23 20:06:40.273849|

25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for execu

# STEP 4: Build GOLD Layer (15 Vertical Transformations)

In [16]:
print("\n" + "="*60)
print("GOLD 1/15: Population Health Index")
print("="*60)

os.makedirs("./output/gold", exist_ok=True)

gold_population_health_index = silver_regional_demographics \
    .join(silver_health_outcomes, ["state", "zip", "county"], "left") \
    .withColumn("diversity_score",
        (F.col("gender_diversity") + F.col("race_diversity") + F.col("ethnicity_diversity")) / 3) \
    .withColumn("health_burden_score",
        F.col("avg_conditions_per_patient") * F.col("complex_patients_count")) \
    .select(
        "state", "zip", "county",
        "population_count",
        "diversity_score",
        "health_burden_score",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_population_health_index.count()}")
gold_population_health_index.show(5, truncate=False)

gold_population_health_index = add_layer_metadata(
    df=gold_population_health_index,
    layer="gold",
    source_tables=["silver_regional_demographics", "silver_health_outcomes"],
    target_table="gold_population_health_index"
)

# Write immediately!
gold_population_health_index.write.mode("overwrite").parquet("./output/gold/gold_population_health_index")
print(f"✓ PHI written")

# Read back
gold_population_health_index = spark.read.parquet("./output/gold/gold_population_health_index")
print(f"  PHI: {gold_population_health_index.count()}")


GOLD 1/15: Population Health Index


25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 33
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 33
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 33
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 34
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 34
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 34
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 34
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 34
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 34
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for execu

  Regions: 191
+-----+----+------+----------------+---------------+-------------------+--------------------------+
|state|zip |county|population_count|diversity_score|health_burden_score|processed_at              |
+-----+----+------+----------------+---------------+-------------------+--------------------------+
|CT   |NULL|7010  |3               |1.0            |NULL               |2025-11-23 20:06:40.640944|
|GA   |NULL|11260 |1               |1.0            |NULL               |2025-11-23 20:06:40.640944|
|ID   |NULL|13000 |3               |1.0            |NULL               |2025-11-23 20:06:40.640944|
|FL   |NULL|10100 |1               |1.0            |NULL               |2025-11-23 20:06:40.640944|
|IN   |NULL|15170 |1               |1.0            |NULL               |2025-11-23 20:06:40.640944|
+-----+----+------+----------------+---------------+-------------------+--------------------------+
only showing top 5 rows

✓ PHI written
  PHI: 191


25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 37
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 37
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for execu

In [17]:
print("\n" + "="*60)
print("GOLD 2/15: Provider-to-Population Access Ratio")
print("="*60)

gold_access_ratio = silver_regional_demographics \
    .join(silver_provider_density, ["state", "zip", "county"], "left") \
    .withColumn("providers_per_1000",
        (F.col("total_providers") / F.col("population_count")) * 1000) \
    .withColumn("specialties_per_1000",
        (F.col("specialty_diversity") / F.col("population_count")) * 1000) \
    .withColumn("care_sites_per_1000",
        (F.col("care_sites_count") / F.col("population_count")) * 1000) \
    .select(
        "state", "zip", "county",
        "providers_per_1000",
        "specialties_per_1000",
        "care_sites_per_1000",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_access_ratio.write.mode("overwrite").parquet("./output/gold/gold_access_ratio")
print(f"✓ AR written")

# Read back
gold_access_ratio = spark.read.parquet("./output/gold/gold_access_ratio")
print(f"  AR: {gold_access_ratio.count()}")

print(f"  Regions: {gold_access_ratio.count()}")
gold_access_ratio.show(5, truncate=False)

25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 41
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 41
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 41
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 42
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 42
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 42
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 42
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 43
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 43
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for execu


GOLD 2/15: Provider-to-Population Access Ratio
✓ AR written
  AR: 191
  Regions: 191
+-----+----+------+------------------+--------------------+-------------------+--------------------------+
|state|zip |county|providers_per_1000|specialties_per_1000|care_sites_per_1000|processed_at              |
+-----+----+------+------------------+--------------------+-------------------+--------------------------+
|CT   |NULL|7010  |NULL              |NULL                |NULL               |2025-11-23 20:06:40.877413|
|GA   |NULL|11260 |NULL              |NULL                |NULL               |2025-11-23 20:06:40.877413|
|ID   |NULL|13000 |NULL              |NULL                |NULL               |2025-11-23 20:06:40.877413|
|FL   |NULL|10100 |NULL              |NULL                |NULL               |2025-11-23 20:06:40.877413|
|IN   |NULL|15170 |NULL              |NULL                |NULL               |2025-11-23 20:06:40.877413|
+-----+----+------+------------------+--------------------

25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 44
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 44
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 44
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 44
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 45
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 45
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 45
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 46
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 46
25/11/23 20:06:40 ERROR ContextFactory: Query execution is null: can't emit event for execu

In [18]:
print("\n" + "="*60)
print("GOLD 3/15: Service Utilization Intensity")
print("="*60)

gold_utilization_intensity = silver_service_utilization \
    .join(silver_regional_demographics, ["state", "zip", "county"], "left") \
    .withColumn("events_per_capita",
        F.col("total_events") / F.col("population_count")) \
    .withColumn("condition_intensity",
        F.col("condition_events") / F.col("population_count")) \
    .withColumn("procedure_intensity",
        F.col("procedure_events") / F.col("population_count")) \
    .withColumn("drug_intensity",
        F.col("drug_events") / F.col("population_count")) \
    .select(
        "state", "zip", "county",
        "events_per_capita",
        "condition_intensity",
        "procedure_intensity",
        "drug_intensity",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_utilization_intensity.write.mode("overwrite").parquet("./output/gold/gold_utilization_intensity")
print(f"✓ UI written")

# Read back
gold_utilization_intensity = spark.read.parquet("./output/gold/gold_utilization_intensity")
print(f"  UI: {gold_utilization_intensity.count()}")

print(f"  Regions: {gold_utilization_intensity.count()}")
gold_utilization_intensity.show(5, truncate=False)

25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 49
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 49
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 49
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 50
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 50
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 50



GOLD 3/15: Service Utilization Intensity
✓ UI written
  UI: 3


25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 51
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 51
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 51
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 52
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 52
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 52
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 53
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 53


  Regions: 3
+-----+----+------+-----------------+-------------------+-------------------+--------------+--------------------------+
|state|zip |county|events_per_capita|condition_intensity|procedure_intensity|drug_intensity|processed_at              |
+-----+----+------+-----------------+-------------------+-------------------+--------------+--------------------------+
|NULL |NULL|0     |NULL             |NULL               |NULL               |NULL          |2025-11-23 20:06:41.119575|
|NULL |NULL|0     |NULL             |NULL               |NULL               |NULL          |2025-11-23 20:06:41.119575|
|NULL |NULL|0     |NULL             |NULL               |NULL               |NULL          |2025-11-23 20:06:41.119575|
+-----+----+------+-----------------+-------------------+-------------------+--------------+--------------------------+



In [19]:
print("\n" + "="*60)
print("GOLD 4/15: Population Vulnerability Score")
print("="*60)

if silver_sdoh_indicators is not None:
    gold_vulnerability_score = silver_sdoh_indicators \
        .withColumn("vulnerability_index",
            (F.col("dual_eligible_ratio") * 0.4 +
             (F.col("qmb_only_count") / F.col("total_dual_eligible")) * 0.3 +
             (F.col("other_dual_count") / F.col("total_dual_eligible")) * 0.3)) \
        .select(
            F.col("State_Abbr").alias("state"),
            F.col("County_Name").alias("county"),
            "vulnerability_index",
            "total_dual_eligible",
            "dual_eligible_ratio",
            F.current_timestamp().alias("processed_at")
        )
    
    # Write immediately!
    gold_vulnerability_score.write.mode("overwrite").parquet("./output/gold/gold_vulnerability_score")
    print(f"✓ VS written")

    # Read back
    gold_vulnerability_score = spark.read.parquet("./output/gold/gold_vulnerability_score")
    print(f"  VS: {gold_vulnerability_score.count()}")

    print(f"  Regions: {gold_vulnerability_score.count()}")
    gold_vulnerability_score.show(5, truncate=False)
else:
    print("  ✗ Skipping - SDOH data not available")
    gold_vulnerability_score = None


GOLD 4/15: Population Vulnerability Score
✓ VS written
  VS: 125
  Regions: 125
+-----+--------+-------------------+-------------------+-------------------+--------------------------+
|state|county  |vulnerability_index|total_dual_eligible|dual_eligible_ratio|processed_at              |
+-----+--------+-------------------+-------------------+-------------------+--------------------------+
|AL   |BULLOCK |0.26703169188585985|4172               |0.35945363048166784|2025-11-23 20:06:41.347671|
|AL   |CHOCTAW |0.26987898135792665|7299               |0.3832876712328767 |2025-11-23 20:06:41.347671|
|AL   |Crenshaw|0.24562289881760344|8070               |0.30801635485070006|2025-11-23 20:06:41.347671|
|AL   |Dale    |0.2597693699102335 |18830              |0.35691147575805854|2025-11-23 20:06:41.347671|
|AL   |Elmore  |0.24558701098499963|21940              |0.3024474727678775 |2025-11-23 20:06:41.347671|
+-----+--------+-------------------+-------------------+-------------------+-----------

25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 55
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 55
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 55
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 56
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 56
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 56
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 57
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 57
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 58
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for execu

In [20]:
print("\n" + "="*60)
print("GOLD 5/15: Quality-Adjusted Access Score")
print("="*60)

if silver_facility_access is not None:
    gold_quality_access = silver_facility_access \
        .join(silver_regional_demographics, 
              [silver_facility_access.state == silver_regional_demographics.state,
               silver_facility_access.zip_code == silver_regional_demographics.zip], "left") \
        .withColumn("quality_adjusted_access",
            (F.col("hospital_count") / F.col("population_count") * 10000) * 
            F.coalesce(F.col("avg_hospital_rating"), F.lit(3))) \
        .select(
            silver_facility_access.state,
            silver_facility_access.zip_code.alias("zip"),
            silver_facility_access.county_name.alias("county"),
            "quality_adjusted_access",
            "hospital_count",
            "avg_hospital_rating",
            F.current_timestamp().alias("processed_at")
        )
    
    # Write immediately!
    gold_quality_access.write.mode("overwrite").parquet("./output/gold/gold_quality_access")
    print(f"✓ QA written")

    # Read back
    gold_quality_access = spark.read.parquet("./output/gold/gold_quality_access")
    print(f"  QA: {gold_quality_access.count()}")

    print(f"  Regions: {gold_quality_access.count()}")
    gold_quality_access.show(5, truncate=False)
else:
    print("  ✗ Skipping - Facility data not available")
    gold_quality_access = None


GOLD 5/15: Quality-Adjusted Access Score
✓ QA written
  QA: 984
  Regions: 984
+-----+-----+---------+-----------------------+--------------+-------------------+--------------------------+
|state|zip  |county   |quality_adjusted_access|hospital_count|avg_hospital_rating|processed_at              |
+-----+-----+---------+-----------------------+--------------+-------------------+--------------------------+
|OH   |43110|Franklin |NULL                   |1             |NULL               |2025-11-23 20:06:41.624635|
|NE   |68122|Douglas  |NULL                   |1             |NULL               |2025-11-23 20:06:41.624635|
|CA   |92225|Riverside|NULL                   |1             |NULL               |2025-11-23 20:06:41.624635|
|KS   |67950|Morton   |NULL                   |1             |NULL               |2025-11-23 20:06:41.624635|
|CA   |95338|Mariposa |NULL                   |1             |4.0                |2025-11-23 20:06:41.624635|
+-----+-----+---------+-----------------

25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 59
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 59
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 59
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 60
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 60
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 60
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 61
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 61


In [21]:
print("\n" + "="*60)
print("GOLD 6/15: Healthcare Service Diversity Index")
print("="*60)

gold_service_diversity = silver_provider_density \
    .withColumn("service_diversity_index",
        (F.col("specialty_diversity") * 0.6 + F.col("care_sites_count") * 0.4) / 10) \
    .select(
        "state", "zip", "county",
        "service_diversity_index",
        "specialty_diversity",
        "care_sites_count",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_service_diversity.write.mode("overwrite").parquet("./output/gold/gold_service_diversity")
print(f"✓ SD written")

# Read back
gold_service_diversity = spark.read.parquet("./output/gold/gold_service_diversity")
print(f"  SD: {gold_service_diversity.count()}")

print(f"  Regions: {gold_service_diversity.count()}")
gold_service_diversity.show(5, truncate=False)


GOLD 6/15: Healthcare Service Diversity Index
✓ SD written
  SD: 1
  Regions: 1
+-----+----+------+-----------------------+-------------------+----------------+--------------------------+
|state|zip |county|service_diversity_index|specialty_diversity|care_sites_count|processed_at              |
+-----+----+------+-----------------------+-------------------+----------------+--------------------------+
|NULL |NULL|NULL  |6.4799999999999995     |0                  |162             |2025-11-23 20:06:41.850014|
+-----+----+------+-----------------------+-------------------+----------------+--------------------------+



25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 63
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 63
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 63
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 64
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 64
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 64
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 65
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 65
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 66
25/11/23 20:06:41 ERROR ContextFactory: Query execution is null: can't emit event for execu

In [22]:
print("\n" + "="*60)
print("GOLD 7/15: Multi-Dimensional Care Accessibility")
print("="*60)

gold_care_accessibility = gold_access_ratio \
    .join(gold_service_diversity, ["state", "zip", "county"], "left") \
    .withColumn("accessibility_composite",
        (F.coalesce(F.col("providers_per_1000"), F.lit(0)) * 0.4 +
         F.coalesce(F.col("specialties_per_1000"), F.lit(0)) * 10 * 0.3 +
         F.coalesce(F.col("service_diversity_index"), F.lit(0)) * 0.3)) \
    .select(
        "state", "zip", "county",
        "accessibility_composite",
        "providers_per_1000",
        "specialties_per_1000",
        "service_diversity_index",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_care_accessibility.write.mode("overwrite").parquet("./output/gold/gold_care_accessibility")
print(f"✓ CA written")

# Read back
gold_care_accessibility = spark.read.parquet("./output/gold/gold_care_accessibility")
print(f"  CA: {gold_care_accessibility.count()}")

print(f"  Regions: {gold_care_accessibility.count()}")
gold_care_accessibility.show(5, truncate=False)


GOLD 7/15: Multi-Dimensional Care Accessibility


25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 67
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 67
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 67
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 68
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 68
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 68
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 69
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 69


✓ CA written
  CA: 191
  Regions: 191
+-----+----+------+-----------------------+------------------+--------------------+-----------------------+--------------------------+
|state|zip |county|accessibility_composite|providers_per_1000|specialties_per_1000|service_diversity_index|processed_at              |
+-----+----+------+-----------------------+------------------+--------------------+-----------------------+--------------------------+
|CT   |NULL|7010  |0.0                    |NULL              |NULL                |NULL                   |2025-11-23 20:06:42.042441|
|GA   |NULL|11260 |0.0                    |NULL              |NULL                |NULL                   |2025-11-23 20:06:42.042441|
|ID   |NULL|13000 |0.0                    |NULL              |NULL                |NULL                   |2025-11-23 20:06:42.042441|
|FL   |NULL|10100 |0.0                    |NULL              |NULL                |NULL                   |2025-11-23 20:06:42.042441|
|IN   |NULL|15170

In [23]:
print("\n" + "="*60)
print("GOLD 8/15: Healthcare Need vs Supply Gap")
print("="*60)

gold_need_supply_gap = gold_population_health_index \
    .join(gold_care_accessibility, ["state", "zip", "county"], "left") \
    .withColumn("need_score",
        F.col("health_burden_score") / (F.col("diversity_score") + 1)) \
    .withColumn("supply_score",
        F.coalesce(F.col("accessibility_composite"), F.lit(0))) \
    .withColumn("gap_magnitude",
        F.col("need_score") - F.col("supply_score")) \
    .select(
        "state", "zip", "county",
        "need_score",
        "supply_score",
        "gap_magnitude",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_need_supply_gap.write.mode("overwrite").parquet("./output/gold/gold_need_supply_gap")
print(f"✓ NSG written")

# Read back
gold_need_supply_gap = spark.read.parquet("./output/gold/gold_need_supply_gap")
print(f"  NSG: {gold_need_supply_gap.count()}")

print(f"  Regions: {gold_need_supply_gap.count()}")
gold_need_supply_gap.show(5, truncate=False)


GOLD 8/15: Healthcare Need vs Supply Gap
✓ NSG written
  NSG: 191
  Regions: 191
+-----+----+------+----------+------------+-------------+--------------------------+
|state|zip |county|need_score|supply_score|gap_magnitude|processed_at              |
+-----+----+------+----------+------------+-------------+--------------------------+
|CT   |NULL|7010  |NULL      |0.0         |NULL         |2025-11-23 20:06:42.259223|
|GA   |NULL|11260 |NULL      |0.0         |NULL         |2025-11-23 20:06:42.259223|
|ID   |NULL|13000 |NULL      |0.0         |NULL         |2025-11-23 20:06:42.259223|
|FL   |NULL|10100 |NULL      |0.0         |NULL         |2025-11-23 20:06:42.259223|
|IN   |NULL|15170 |NULL      |0.0         |NULL         |2025-11-23 20:06:42.259223|
+-----+----+------+----------+------------+-------------+--------------------------+
only showing top 5 rows



25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 71
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 71
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 71
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 72
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 72
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 72
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 73
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 73


In [24]:
print("\n" + "="*60)
print("GOLD 9/15: Underserved Area Classification")
print("="*60)

window_spec = W.partitionBy()

gold_underserved_flag = gold_need_supply_gap \
    .withColumn("gap_percentile",
        F.percent_rank().over(window_spec.orderBy(F.col("gap_magnitude").desc()))) \
    .withColumn("underserved_severity",
        F.when(F.col("gap_percentile") < 0.1, "Critical")
         .when(F.col("gap_percentile") < 0.3, "High")
         .when(F.col("gap_percentile") < 0.6, "Moderate")
         .otherwise("Low")) \
    .select(
        "state", "zip", "county",
        "gap_magnitude",
        "gap_percentile",
        "underserved_severity",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_underserved_flag.write.mode("overwrite").parquet("./output/gold/gold_underserved_flag")
print(f"✓ UF written")

# Read back
gold_underserved_flag = spark.read.parquet("./output/gold/gold_underserved_flag")
print(f"  UF: {gold_underserved_flag.count()}")

print(f"  Regions: {gold_underserved_flag.count()}")
gold_underserved_flag.show(5, truncate=False)

25/11/23 20:06:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



GOLD 9/15: Underserved Area Classification
✓ UF written


25/11/23 20:06:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 20:06:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 20:06:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 20:06:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 75
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 75
25/11/23 20:06:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 75
25/11/23 20:06:42 ERROR ContextFactor

  UF: 191
  Regions: 191
+-----+----+------+-------------+--------------+--------------------+--------------------------+
|state|zip |county|gap_magnitude|gap_percentile|underserved_severity|processed_at              |
+-----+----+------+-------------+--------------+--------------------+--------------------------+
|CT   |NULL|7010  |NULL         |0.0           |Critical            |2025-11-23 20:06:42.520142|
|GA   |NULL|11260 |NULL         |0.0           |Critical            |2025-11-23 20:06:42.520142|
|ID   |NULL|13000 |NULL         |0.0           |Critical            |2025-11-23 20:06:42.520142|
|FL   |NULL|10100 |NULL         |0.0           |Critical            |2025-11-23 20:06:42.520142|
|IN   |NULL|15170 |NULL         |0.0           |Critical            |2025-11-23 20:06:42.520142|
+-----+----+------+-------------+--------------+--------------------+--------------------------+
only showing top 5 rows



In [25]:
print("\n" + "="*60)
print("GOLD 10/15: Utilization Efficiency Score")
print("="*60)

gold_utilization_efficiency = gold_utilization_intensity \
    .join(gold_care_accessibility, ["state", "zip", "county"], "left") \
    .withColumn("efficiency_ratio",
        F.col("events_per_capita") / 
        (F.coalesce(F.col("accessibility_composite"), F.lit(1)) + 0.01)) \
    .withColumn("normalized_efficiency",
        F.when(F.col("efficiency_ratio") > 2, F.lit("Over-utilized"))
         .when(F.col("efficiency_ratio") > 0.5, F.lit("Balanced"))
         .otherwise("Under-utilized")) \
    .select(
        "state", "zip", "county",
        "efficiency_ratio",
        "normalized_efficiency",
        "events_per_capita",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_utilization_efficiency.write.mode("overwrite").parquet("./output/gold/gold_utilization_efficiency")
print(f"✓ UE written")

# Read back
gold_utilization_efficiency = spark.read.parquet("./output/gold/gold_utilization_efficiency")
print(f"  UE: {gold_utilization_efficiency.count()}")

print(f"  Regions: {gold_utilization_efficiency.count()}")
gold_utilization_efficiency.show(5, truncate=False)


GOLD 10/15: Utilization Efficiency Score
✓ UE written
  UE: 3
  Regions: 3
+-----+----+------+----------------+---------------------+-----------------+--------------------------+
|state|zip |county|efficiency_ratio|normalized_efficiency|events_per_capita|processed_at              |
+-----+----+------+----------------+---------------------+-----------------+--------------------------+
|NULL |NULL|0     |NULL            |Under-utilized       |NULL             |2025-11-23 20:06:42.766361|
|NULL |NULL|0     |NULL            |Under-utilized       |NULL             |2025-11-23 20:06:42.766361|
|NULL |NULL|0     |NULL            |Under-utilized       |NULL             |2025-11-23 20:06:42.766361|
+-----+----+------+----------------+---------------------+-----------------+--------------------------+



In [26]:
print("\n" + "="*60)
print("GOLD 11/15: Regional Inequity Magnitude")
print("="*60)

state_avg = gold_care_accessibility \
    .groupBy("state") \
    .agg(F.avg("accessibility_composite").alias("state_avg_access"))

gold_regional_inequity = gold_care_accessibility \
    .join(state_avg, "state", "left") \
    .withColumn("access_deviation",
        F.col("accessibility_composite") - F.col("state_avg_access")) \
    .withColumn("inequity_magnitude",
        F.abs(F.col("access_deviation")) / (F.col("state_avg_access") + 0.01)) \
    .select(
        "state", "zip", "county",
        "access_deviation",
        "inequity_magnitude",
        "state_avg_access",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_regional_inequity.write.mode("overwrite").parquet("./output/gold/gold_regional_inequity")
print(f"✓ RI written")

# Read back
gold_regional_inequity = spark.read.parquet("./output/gold/gold_regional_inequity")
print(f"  RI: {gold_regional_inequity.count()}")

print(f"  Regions: {gold_regional_inequity.count()}")
gold_regional_inequity.show(5, truncate=False)


GOLD 11/15: Regional Inequity Magnitude
✓ RI written
  RI: 191
  Regions: 191
+-----+----+------+----------------+------------------+----------------+--------------------------+
|state|zip |county|access_deviation|inequity_magnitude|state_avg_access|processed_at              |
+-----+----+------+----------------+------------------+----------------+--------------------------+
|CT   |NULL|7010  |0.0             |0.0               |0.0             |2025-11-23 20:06:42.993753|
|GA   |NULL|11260 |0.0             |0.0               |0.0             |2025-11-23 20:06:42.993753|
|ID   |NULL|13000 |0.0             |0.0               |0.0             |2025-11-23 20:06:42.993753|
|FL   |NULL|10100 |0.0             |0.0               |0.0             |2025-11-23 20:06:42.993753|
|IN   |NULL|15170 |0.0             |0.0               |0.0             |2025-11-23 20:06:42.993753|
+-----+----+------+----------------+------------------+----------------+--------------------------+
only showing top 5 ro

25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 83
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 83
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 83
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 84
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 84
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 84
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 85
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 85
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 86
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for execu

In [27]:
print("\n" + "="*60)
print("GOLD 12/15: Compound Disadvantage Index")
print("="*60)

base_disadvantage = gold_underserved_flag \
    .join(gold_utilization_efficiency, ["state", "zip", "county"], "left")

if gold_vulnerability_score is not None:
    gold_compound_disadvantage = base_disadvantage \
        .join(gold_vulnerability_score,
              [base_disadvantage.state == gold_vulnerability_score.state,
               base_disadvantage.county == gold_vulnerability_score.county], "left") \
        .withColumn("compound_index",
            (F.col("gap_percentile") * 0.4 +
             F.when(F.col("normalized_efficiency") == "Under-utilized", 0.3).otherwise(0) +
             F.coalesce(F.col("vulnerability_index"), F.lit(0)) * 0.3)) \
        .select(
            base_disadvantage.state,
            base_disadvantage.zip,
            base_disadvantage.county,
            "compound_index",
            "underserved_severity",
            "normalized_efficiency",
            F.current_timestamp().alias("processed_at")
        )
else:
    gold_compound_disadvantage = base_disadvantage \
        .withColumn("compound_index",
            (F.col("gap_percentile") * 0.6 +
             F.when(F.col("normalized_efficiency") == "Under-utilized", 0.4).otherwise(0))) \
        .select(
            "state", "zip", "county",
            "compound_index",
            "underserved_severity",
            "normalized_efficiency",
            F.current_timestamp().alias("processed_at")
        )
    
# Write immediately!
gold_compound_disadvantage.write.mode("overwrite").parquet("./output/gold/gold_compound_disadvantage")
print(f"✓ CD written")

# Read back
gold_compound_disadvantage = spark.read.parquet("./output/gold/gold_compound_disadvantage")
print(f"  CD: {gold_compound_disadvantage.count()}")

print(f"  Regions: {gold_compound_disadvantage.count()}")
gold_compound_disadvantage.show(5, truncate=False)


GOLD 12/15: Compound Disadvantage Index
✓ CD written
  CD: 191
  Regions: 191
+-----+----+------+--------------+--------------------+---------------------+--------------------------+
|state|zip |county|compound_index|underserved_severity|normalized_efficiency|processed_at              |
+-----+----+------+--------------+--------------------+---------------------+--------------------------+
|CT   |NULL|7010  |0.0           |Critical            |NULL                 |2025-11-23 20:06:43.263844|
|GA   |NULL|11260 |0.0           |Critical            |NULL                 |2025-11-23 20:06:43.263844|
|ID   |NULL|13000 |0.0           |Critical            |NULL                 |2025-11-23 20:06:43.263844|
|FL   |NULL|10100 |0.0           |Critical            |NULL                 |2025-11-23 20:06:43.263844|
|IN   |NULL|15170 |0.0           |Critical            |NULL                 |2025-11-23 20:06:43.263844|
+-----+----+------+--------------+--------------------+---------------------+----

In [28]:
print("\n" + "="*60)
print("GOLD 13/15: Temporal Stability Assessment")
print("="*60)

gold_temporal_stability = gold_utilization_intensity \
    .withColumn("service_variance",
        F.pow(F.col("condition_intensity") - F.col("drug_intensity"), 2) +
        F.pow(F.col("procedure_intensity") - F.col("drug_intensity"), 2)) \
    .withColumn("stability_score",
        F.lit(1) / (F.col("service_variance") + 0.01)) \
    .select(
        "state", "zip", "county",
        "stability_score",
        "service_variance",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_temporal_stability.write.mode("overwrite").parquet("./output/gold/gold_temporal_stability")
print(f"✓ TS written")

# Read back
gold_temporal_stability = spark.read.parquet("./output/gold/gold_temporal_stability")
print(f"  TS: {gold_temporal_stability.count()}")

print(f"  Regions: {gold_temporal_stability.count()}")
gold_temporal_stability.show(5, truncate=False)


GOLD 13/15: Temporal Stability Assessment
✓ TS written
  TS: 3
  Regions: 3
+-----+----+------+---------------+----------------+--------------------------+
|state|zip |county|stability_score|service_variance|processed_at              |
+-----+----+------+---------------+----------------+--------------------------+
|NULL |NULL|0     |NULL           |NULL            |2025-11-23 20:06:43.499405|
|NULL |NULL|0     |NULL           |NULL            |2025-11-23 20:06:43.499405|
|NULL |NULL|0     |NULL           |NULL            |2025-11-23 20:06:43.499405|
+-----+----+------+---------------+----------------+--------------------------+



In [29]:
print("\n" + "="*60)
print("GOLD 14/15: Integrated Regional Ranking")
print("="*60)

window_national = W.partitionBy()

gold_integrated_ranking = gold_care_accessibility \
    .join(gold_need_supply_gap, ["state", "zip", "county"], "left") \
    .join(gold_compound_disadvantage, ["state", "zip", "county"], "left") \
    .withColumn("composite_score",
        F.coalesce(F.col("accessibility_composite"), F.lit(0)) * 0.4 -
        F.col("gap_magnitude") * 0.3 -
        F.coalesce(F.col("compound_index"), F.lit(0)) * 0.3) \
    .withColumn("national_percentile",
        F.percent_rank().over(window_national.orderBy(F.col("composite_score").desc()))) \
    .withColumn("tier",
        F.when(F.col("national_percentile") < 0.2, "Top 20%")
         .when(F.col("national_percentile") < 0.5, "Above Average")
         .when(F.col("national_percentile") < 0.8, "Below Average")
         .otherwise("Bottom 20%")) \
    .select(
        "state", "zip", "county",
        "composite_score",
        "national_percentile",
        "tier",
        F.current_timestamp().alias("processed_at")
    )

# Write immediately!
gold_integrated_ranking.write.mode("overwrite").parquet("./output/gold/gold_integrated_ranking")
print(f"✓ IR written")

# Read back
gold_integrated_ranking = spark.read.parquet("./output/gold/gold_integrated_ranking")
print(f"  IR: {gold_integrated_ranking.count()}")

print(f"  Regions: {gold_integrated_ranking.count()}")
gold_integrated_ranking.show(5, truncate=False)


GOLD 14/15: Integrated Regional Ranking
✓ IR written
  IR: 191


25/11/23 20:06:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 20:06:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 20:06:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 91
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 91
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 91
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 92
25/11/23 20:06:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 92


  Regions: 191
+-----+----+------+---------------+-------------------+-------+--------------------------+
|state|zip |county|composite_score|national_percentile|tier   |processed_at              |
+-----+----+------+---------------+-------------------+-------+--------------------------+
|CT   |NULL|7010  |NULL           |0.0                |Top 20%|2025-11-23 20:06:43.705345|
|GA   |NULL|11260 |NULL           |0.0                |Top 20%|2025-11-23 20:06:43.705345|
|ID   |NULL|13000 |NULL           |0.0                |Top 20%|2025-11-23 20:06:43.705345|
|FL   |NULL|10100 |NULL           |0.0                |Top 20%|2025-11-23 20:06:43.705345|
|IN   |NULL|15170 |NULL           |0.0                |Top 20%|2025-11-23 20:06:43.705345|
+-----+----+------+---------------+-------------------+-------+--------------------------+
only showing top 5 rows



In [30]:
print("\n" + "="*60)
print("GOLD 15/15: Predictive Deterioration Risk")
print("="*60)

gold_predictive_risk = gold_need_supply_gap \
    .join(gold_temporal_stability, ["state", "zip", "county"], "left") \
    .join(gold_compound_disadvantage, ["state", "zip", "county"], "left") \
    .withColumn("risk_score",
        F.col("gap_magnitude") * 0.4 +
        (F.lit(1) - F.coalesce(F.col("stability_score"), F.lit(0.5))) * 0.3 +
        F.coalesce(F.col("compound_index"), F.lit(0)) * 0.3) \
    .withColumn("risk_category",
        F.when(F.col("risk_score") > 0.7, "High Risk")
         .when(F.col("risk_score") > 0.4, "Moderate Risk")
         .otherwise("Low Risk")) \
    .select(
        "state", "zip", "county",
        "risk_score",
        "risk_category",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_predictive_risk.count()}")
gold_predictive_risk.show(5, truncate=False)

gold_predictive_risk = add_layer_metadata(
    df=gold_predictive_risk,
    layer="gold",
    source_tables=["gold_need_supply_gap", "gold_temporal_stability", "gold_compound_disadvantage"],
    target_table="gold_predictive_risk"
)

# Write immediately!
gold_predictive_risk.write.mode("overwrite").parquet("./output/gold/gold_predictive_risk")
print(f"✓ PR written")

# Read back
gold_predictive_risk = spark.read.parquet("./output/gold/gold_predictive_risk")
print(f"  PR: {gold_predictive_risk.count()}")

print("\n✓ Gold Layer complete (15 transformations)")


GOLD 15/15: Predictive Deterioration Risk
  Regions: 191
+-----+----+------+----------+-------------+--------------------------+
|state|zip |county|risk_score|risk_category|processed_at              |
+-----+----+------+----------+-------------+--------------------------+
|CT   |NULL|7010  |NULL      |Low Risk     |2025-11-23 20:06:44.008067|
|GA   |NULL|11260 |NULL      |Low Risk     |2025-11-23 20:06:44.008067|
|ID   |NULL|13000 |NULL      |Low Risk     |2025-11-23 20:06:44.008067|
|FL   |NULL|10100 |NULL      |Low Risk     |2025-11-23 20:06:44.008067|
|IN   |NULL|15170 |NULL      |Low Risk     |2025-11-23 20:06:44.008067|
+-----+----+------+----------+-------------+--------------------------+
only showing top 5 rows

✓ PR written
  PR: 191

✓ Gold Layer complete (15 transformations)


# STEP 5: Final Metrics (4 Top-Level KPIs)

In [31]:
print("\n" + "="*60)
print("METRIC 1/4: Health Equity Index by Region")
print("="*60)

metric_health_equity_index = gold_integrated_ranking \
    .join(gold_care_accessibility, ["state", "zip", "county"], "left") \
    .join(gold_population_health_index, ["state", "zip", "county"], "left") \
    .withColumn("equity_index",
        (F.col("composite_score") * 0.5 +
         (F.lit(1) - F.col("national_percentile")) * 0.3 +
         (F.lit(1) / (F.col("health_burden_score") + 1)) * 0.2) * 100) \
    .withColumn("equity_grade",
        F.when(F.col("equity_index") >= 80, "A")
         .when(F.col("equity_index") >= 60, "B")
         .when(F.col("equity_index") >= 40, "C")
         .when(F.col("equity_index") >= 20, "D")
         .otherwise("F")) \
    .select(
        "state", "zip", "county",
        F.round("equity_index", 2).alias("health_equity_index"),
        "equity_grade",
        "tier",
        F.current_timestamp().alias("calculated_at")
    )

print(f"  Total regions: {metric_health_equity_index.count()}")
metric_health_equity_index.orderBy(F.desc("health_equity_index")).show(10, truncate=False)

print("\nEquity Grade Distribution:")
metric_health_equity_index.groupBy("equity_grade").count().orderBy("equity_grade").show()

# ADD METADATA
metric_health_equity_index = add_layer_metadata(
    df=metric_health_equity_index,
    layer="gold",
    source_tables=["gold_integrated_ranking", "gold_care_accessibility", "gold_population_health_index"],
    target_table="metric_health_equity_index"
)

# Write immediately!
metric_health_equity_index.write.mode("overwrite").parquet("./output/gold/metric_health_equity_index")
print(f"✓ MHEI written")

# Read back
metric_health_equity_index = spark.read.parquet("./output/gold/metric_health_equity_index")
print(f"  MHEI: {metric_health_equity_index.count()}")


METRIC 1/4: Health Equity Index by Region
  Total regions: 191
+-----+----+------+-------------------+------------+-------+--------------------------+
|state|zip |county|health_equity_index|equity_grade|tier   |calculated_at             |
+-----+----+------+-------------------+------------+-------+--------------------------+
|CT   |NULL|7010  |NULL               |F           |Top 20%|2025-11-23 20:06:44.337156|
|GA   |NULL|11260 |NULL               |F           |Top 20%|2025-11-23 20:06:44.337156|
|ID   |NULL|13000 |NULL               |F           |Top 20%|2025-11-23 20:06:44.337156|
|FL   |NULL|10100 |NULL               |F           |Top 20%|2025-11-23 20:06:44.337156|
|IN   |NULL|15170 |NULL               |F           |Top 20%|2025-11-23 20:06:44.337156|
|FL   |NULL|10020 |NULL               |F           |Top 20%|2025-11-23 20:06:44.337156|
|IL   |NULL|14141 |NULL               |F           |Top 20%|2025-11-23 20:06:44.337156|
|CO   |NULL|6220  |NULL               |F           |Top 

25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 99
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 99
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 99
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 100
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 100
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 100
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 101
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 101
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 102
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for

✓ MHEI written
  MHEI: 191


25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 108
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 108
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 108
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 108
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 108


In [32]:
print("\n" + "="*60)
print("METRIC 2/4: Care Desert Score")
print("="*60)

metric_care_desert = gold_underserved_flag \
    .join(gold_access_ratio, ["state", "zip", "county"], "left") \
    .join(gold_service_diversity, ["state", "zip", "county"], "left") \
    .withColumn("desert_score",
        (F.col("gap_percentile") * 0.4 +
         (F.lit(1) / (F.coalesce(F.col("providers_per_1000"), F.lit(0.1)) + 0.1)) * 0.3 +
         (F.lit(1) / (F.coalesce(F.col("service_diversity_index"), F.lit(0.1)) + 0.1)) * 0.3) * 100) \
    .withColumn("desert_classification",
        F.when(F.col("desert_score") >= 80, "Severe Desert")
         .when(F.col("desert_score") >= 60, "Moderate Desert")
         .when(F.col("desert_score") >= 40, "Mild Desert")
         .otherwise("Adequate Coverage")) \
    .select(
        "state", "zip", "county",
        F.round("desert_score", 2).alias("care_desert_score"),
        "desert_classification",
        "underserved_severity",
        F.current_timestamp().alias("calculated_at")
    )

print(f"  Total regions: {metric_care_desert.count()}")
metric_care_desert.orderBy(F.desc("care_desert_score")).show(10, truncate=False)

print("\nDesert Classification Distribution:")
metric_care_desert.groupBy("desert_classification").count().orderBy("desert_classification").show()

# ADD METADATA
metric_care_desert = add_layer_metadata(
    df=metric_care_desert,
    layer="gold",
    source_tables=["gold_underserved_flag", "gold_access_ratio", "gold_service_diversity"],
    target_table="metric_care_desert_score"
)

# Write immediately!
metric_care_desert.write.mode("overwrite").parquet("./output/gold/metric_care_desert")
print(f"✓ MHEI written")

# Read back
metric_care_desert = spark.read.parquet("./output/gold/metric_care_desert")
print(f"  MHEI: {metric_care_desert.count()}")


METRIC 2/4: Care Desert Score
  Total regions: 191


25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 109
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 109
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 109
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 109
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 110
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 110
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 110
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 111
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 111
25/11/23 20:06:44 ERROR ContextFactory: Query execution is null: can't emit event 

+-----+----+------+-----------------+---------------------+--------------------+--------------------------+
|state|zip |county|care_desert_score|desert_classification|underserved_severity|calculated_at             |
+-----+----+------+-----------------+---------------------+--------------------+--------------------------+
|CT   |NULL|7010  |300.0            |Severe Desert        |Critical            |2025-11-23 20:06:44.804274|
|GA   |NULL|11260 |300.0            |Severe Desert        |Critical            |2025-11-23 20:06:44.804274|
|ID   |NULL|13000 |300.0            |Severe Desert        |Critical            |2025-11-23 20:06:44.804274|
|FL   |NULL|10100 |300.0            |Severe Desert        |Critical            |2025-11-23 20:06:44.804274|
|IN   |NULL|15170 |300.0            |Severe Desert        |Critical            |2025-11-23 20:06:44.804274|
|FL   |NULL|10020 |300.0            |Severe Desert        |Critical            |2025-11-23 20:06:44.804274|
|IL   |NULL|14141 |300.0    

In [33]:
print("\n" + "="*60)
print("METRIC 3/4: Social Determinants Impact")
print("="*60)

if gold_vulnerability_score is not None:
    metric_sdoh_impact = gold_vulnerability_score \
        .join(gold_compound_disadvantage,
              [gold_vulnerability_score.state == gold_compound_disadvantage.state,
               gold_vulnerability_score.county == gold_compound_disadvantage.county], "left") \
        .withColumn("sdoh_impact_score",
            (F.col("vulnerability_index") * 0.5 +
             F.coalesce(F.col("compound_index"), F.lit(0)) * 0.3 +
             F.col("dual_eligible_ratio") * 0.2) * 100) \
        .withColumn("impact_level",
            F.when(F.col("sdoh_impact_score") >= 75, "Critical Impact")
             .when(F.col("sdoh_impact_score") >= 50, "High Impact")
             .when(F.col("sdoh_impact_score") >= 25, "Moderate Impact")
             .otherwise("Low Impact")) \
        .select(
            gold_vulnerability_score.state,
            gold_vulnerability_score.county,
            F.round("sdoh_impact_score", 2).alias("sdoh_impact_score"),
            "impact_level",
            "vulnerability_index",
            "total_dual_eligible",
            F.current_timestamp().alias("calculated_at")
        )
    
    print(f"  Total regions: {metric_sdoh_impact.count()}")
    metric_sdoh_impact.orderBy(F.desc("sdoh_impact_score")).show(10, truncate=False)
    
    print("\nImpact Level Distribution:")
    metric_sdoh_impact.groupBy("impact_level").count().orderBy("impact_level").show()

    # ADD METADATA
    metric_sdoh_impact = add_layer_metadata(
        df=metric_sdoh_impact,
        layer="gold",
        source_tables=["gold_vulnerability_score", "gold_compound_disadvantage"],
        target_table="metric_sdoh_impact_score"
    )

    # Write immediately!
    metric_sdoh_impact.write.mode("overwrite").parquet("./output/gold/metric_sdoh_impact")
    print(f"✓ MSI written")

    # Read back
    metric_sdoh_impact = spark.read.parquet("./output/gold/metric_sdoh_impact")
    print(f"  MSI: {metric_sdoh_impact.count()}")
else:
    print("  ✗ SDOH Impact metric unavailable - missing dual enrollment data")
    metric_sdoh_impact = None


METRIC 3/4: Social Determinants Impact
  Total regions: 125
+-----+-------+-----------------+------------+-------------------+-------------------+--------------------------+
|state|county |sdoh_impact_score|impact_level|vulnerability_index|total_dual_eligible|calculated_at             |
+-----+-------+-----------------+------------+-------------------+-------------------+--------------------------+
|AL   |SUMTER |24.12            |Low Impact  |0.2972416797277652 |8188               |2025-11-23 20:06:45.163633|
|AL   |Sumter |23.94            |Low Impact  |0.29356354220612196|8636               |2025-11-23 20:06:45.163633|
|AL   |GREENE |22.63            |Low Impact  |0.2794387281470948 |5689               |2025-11-23 20:06:45.163633|
|AL   |DALLAS |22.45            |Low Impact  |0.27756228939210537|35858              |2025-11-23 20:06:45.163633|
|AL   |MARENGO|22.38            |Low Impact  |0.2809412065122073 |10660              |2025-11-23 20:06:45.163633|
|AL   |Marengo|22.32       

25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 113
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 113
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 113
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 113
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 113
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 114
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 114
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 114
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 114
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event 

+------------+-----+
|impact_level|count|
+------------+-----+
|  Low Impact|  125|
+------------+-----+

✓ MSI written
  MSI: 125


25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 118
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 118
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 118
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 118
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 119
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 119
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 119


In [34]:
print("\n" + "="*60)
print("METRIC 4/4: Geographic Disparity Magnitude")
print("="*60)

metric_disparity_magnitude = gold_regional_inequity \
    .join(gold_need_supply_gap, ["state", "zip", "county"], "left") \
    .join(gold_integrated_ranking, ["state", "zip", "county"], "left") \
    .withColumn("disparity_magnitude",
        (F.col("inequity_magnitude") * 0.4 +
         F.abs(F.col("gap_magnitude")) * 0.3 +
         F.col("national_percentile") * 0.3) * 100) \
    .withColumn("disparity_severity",
        F.when(F.col("disparity_magnitude") >= 80, "Extreme Disparity")
         .when(F.col("disparity_magnitude") >= 60, "High Disparity")
         .when(F.col("disparity_magnitude") >= 40, "Moderate Disparity")
         .otherwise("Low Disparity")) \
    .select(
        "state", "zip", "county",
        F.round("disparity_magnitude", 2).alias("geographic_disparity_magnitude"),
        "disparity_severity",
        "tier",
        "access_deviation",
        F.current_timestamp().alias("calculated_at")
    )

print(f"  Total regions: {metric_disparity_magnitude.count()}")
metric_disparity_magnitude.orderBy(F.desc("geographic_disparity_magnitude")).show(10, truncate=False)

print("\nDisparity Severity Distribution:")
metric_disparity_magnitude.groupBy("disparity_severity").count().orderBy("disparity_severity").show()

# ADD METADATA
metric_disparity_magnitude = add_layer_metadata(
    df=metric_disparity_magnitude,
    layer="gold",
    source_tables=["gold_regional_inequity", "gold_need_supply_gap", "gold_integrated_ranking"],
    target_table="metric_geographic_disparity"
)

# Write immediately!
metric_disparity_magnitude.write.mode("overwrite").parquet("./output/gold/metric_disparity_magnitude")
print(f"✓ MDM written")

# Read back
metric_disparity_magnitude = spark.read.parquet("./output/gold/metric_disparity_magnitude")
print(f"  MDM: {metric_disparity_magnitude.count()}")

print("\n✓ All 4 final metrics calculated")


METRIC 4/4: Geographic Disparity Magnitude


25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 120
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 120
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 120


  Total regions: 191
+-----+----+------+------------------------------+------------------+----+----------------+-------------------------+
|state|zip |county|geographic_disparity_magnitude|disparity_severity|tier|access_deviation|calculated_at            |
+-----+----+------+------------------------------+------------------+----+----------------+-------------------------+
|CT   |NULL|7010  |NULL                          |Low Disparity     |NULL|0.0             |2025-11-23 20:06:45.53156|
|GA   |NULL|11260 |NULL                          |Low Disparity     |NULL|0.0             |2025-11-23 20:06:45.53156|
|ID   |NULL|13000 |NULL                          |Low Disparity     |NULL|0.0             |2025-11-23 20:06:45.53156|
|FL   |NULL|10100 |NULL                          |Low Disparity     |NULL|0.0             |2025-11-23 20:06:45.53156|
|IN   |NULL|15170 |NULL                          |Low Disparity     |NULL|0.0             |2025-11-23 20:06:45.53156|
|FL   |NULL|10020 |NULL            

25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 123
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 123
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 123
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 123
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 123


# STEP 6: Build DAG (Bronze -> Silver -> Gold -> Metrics)

In [35]:
print("\n" + "="*60)
print("BUILDING DAG")
print("="*60)

G = nx.DiGraph()

bronze_nodes = [
    "bronze_person", "bronze_location", "bronze_care_site", "bronze_provider",
    "bronze_condition", "bronze_procedure", "bronze_drug", "bronze_obs_period",
    "bronze_dual_enroll", "bronze_hospital_info", "bronze_inpatient"
]

silver_nodes = [
    "silver_regional_demographics", "silver_provider_density",
    "silver_service_utilization", "silver_health_outcomes",
    "silver_sdoh_indicators", "silver_facility_access"
]

gold_nodes = [
    "gold_population_health_index", "gold_access_ratio",
    "gold_utilization_intensity", "gold_vulnerability_score",
    "gold_quality_access", "gold_service_diversity",
    "gold_care_accessibility", "gold_need_supply_gap",
    "gold_underserved_flag", "gold_utilization_efficiency",
    "gold_regional_inequity", "gold_compound_disadvantage",
    "gold_temporal_stability", "gold_integrated_ranking",
    "gold_predictive_risk"
]

metric_nodes = [
    "metric_health_equity_index", "metric_care_desert",
    "metric_sdoh_impact", "metric_disparity_magnitude"
]

for node in bronze_nodes:
    G.add_node(node, layer="bronze", label=node.replace("bronze_", ""))

for node in silver_nodes:
    G.add_node(node, layer="silver", label=node.replace("silver_", ""))

for node in gold_nodes:
    G.add_node(node, layer="gold", label=node.replace("gold_", ""))

for node in metric_nodes:
    G.add_node(node, layer="metric", label=node.replace("metric_", ""))

print(f"  Added {len(bronze_nodes)} bronze nodes")
print(f"  Added {len(silver_nodes)} silver nodes")
print(f"  Added {len(gold_nodes)} gold nodes")
print(f"  Added {len(metric_nodes)} metric nodes")


BUILDING DAG
  Added 11 bronze nodes
  Added 6 silver nodes
  Added 15 gold nodes
  Added 4 metric nodes


In [36]:
print("\n" + "="*60)
print("ADDING EDGES")
print("="*60)

edges = [
    ("bronze_person", "silver_regional_demographics"),
    ("bronze_location", "silver_regional_demographics"),
    ("bronze_provider", "silver_provider_density"),
    ("bronze_care_site", "silver_provider_density"),
    ("bronze_location", "silver_provider_density"),
    ("bronze_person", "silver_service_utilization"),
    ("bronze_location", "silver_service_utilization"),
    ("bronze_condition", "silver_service_utilization"),
    ("bronze_procedure", "silver_service_utilization"),
    ("bronze_drug", "silver_service_utilization"),
    ("bronze_condition", "silver_health_outcomes"),
    ("bronze_person", "silver_health_outcomes"),
    ("bronze_location", "silver_health_outcomes"),
    ("bronze_dual_enroll", "silver_sdoh_indicators"),
    ("bronze_hospital_info", "silver_facility_access"),
    ("silver_regional_demographics", "gold_population_health_index"),
    ("silver_health_outcomes", "gold_population_health_index"),
    ("silver_regional_demographics", "gold_access_ratio"),
    ("silver_provider_density", "gold_access_ratio"),
    ("silver_service_utilization", "gold_utilization_intensity"),
    ("silver_regional_demographics", "gold_utilization_intensity"),
    ("silver_sdoh_indicators", "gold_vulnerability_score"),
    ("silver_facility_access", "gold_quality_access"),
    ("silver_regional_demographics", "gold_quality_access"),
    ("silver_provider_density", "gold_service_diversity"),
    ("gold_access_ratio", "gold_care_accessibility"),
    ("gold_service_diversity", "gold_care_accessibility"),
    ("gold_population_health_index", "gold_need_supply_gap"),
    ("gold_care_accessibility", "gold_need_supply_gap"),
    ("gold_need_supply_gap", "gold_underserved_flag"),
    ("gold_utilization_intensity", "gold_utilization_efficiency"),
    ("gold_care_accessibility", "gold_utilization_efficiency"),
    ("gold_care_accessibility", "gold_regional_inequity"),
    ("gold_underserved_flag", "gold_compound_disadvantage"),
    ("gold_utilization_efficiency", "gold_compound_disadvantage"),
    ("gold_vulnerability_score", "gold_compound_disadvantage"),
    ("gold_utilization_intensity", "gold_temporal_stability"),
    ("gold_care_accessibility", "gold_integrated_ranking"),
    ("gold_need_supply_gap", "gold_integrated_ranking"),
    ("gold_compound_disadvantage", "gold_integrated_ranking"),
    ("gold_need_supply_gap", "gold_predictive_risk"),
    ("gold_temporal_stability", "gold_predictive_risk"),
    ("gold_compound_disadvantage", "gold_predictive_risk"),
    ("gold_integrated_ranking", "metric_health_equity_index"),
    ("gold_care_accessibility", "metric_health_equity_index"),
    ("gold_population_health_index", "metric_health_equity_index"),
    ("gold_underserved_flag", "metric_care_desert"),
    ("gold_access_ratio", "metric_care_desert"),
    ("gold_service_diversity", "metric_care_desert"),
    ("gold_vulnerability_score", "metric_sdoh_impact"),
    ("gold_compound_disadvantage", "metric_sdoh_impact"),
    ("gold_regional_inequity", "metric_disparity_magnitude"),
    ("gold_need_supply_gap", "metric_disparity_magnitude"),
    ("gold_integrated_ranking", "metric_disparity_magnitude")
]

G.add_edges_from(edges)

print(f"  Added {len(edges)} edges")
print(f"  DAG valid: {nx.is_directed_acyclic_graph(G)}")
print(f"  Total nodes: {G.number_of_nodes()}")
print(f"  Total edges: {G.number_of_edges()}")


ADDING EDGES
  Added 54 edges
  DAG valid: True
  Total nodes: 36
  Total edges: 54


In [37]:
print("\n" + "="*60)
print("SAVING DAG AND METADATA")
print("="*60)

dag_file = f"{LOCAL_DATA_DIR}/geographic_health_equity_dag.graphml"
nx.write_graphml(G, dag_file)
print(f"  ✓ Saved DAG: {dag_file}")

rag_data = []
for node_id in G.nodes():
    node_attrs = G.nodes[node_id]
    label = node_attrs.get('label', '')
    layer = node_attrs.get('layer', 'unknown')
    
    in_degree = G.in_degree(node_id)
    out_degree = G.out_degree(node_id)
    
    parents = list(G.predecessors(node_id))
    children = list(G.successors(node_id))
    
    texts = [
        f"{label}",
        f"Layer: {layer}",
        f"Incoming: {in_degree}, Outgoing: {out_degree}"
    ]
    
    if parents:
        texts.append(f"Consumes: {', '.join(parents[:3])}")
    
    if children:
        texts.append(f"Feeds into: {', '.join(children[:3])}")
    
    rag_data.append({
        "id": node_id,
        "texts": texts
    })

rag_file = f"{LOCAL_DATA_DIR}/geographic_health_equity_rag_data.json"
with open(rag_file, 'w') as f:
    json.dump(rag_data, f, indent=2)
print(f"  ✓ Saved RAG data: {rag_file}")

stats = {
    "total_nodes": G.number_of_nodes(),
    "total_edges": G.number_of_edges(),
    "bronze_nodes": len(bronze_nodes),
    "silver_nodes": len(silver_nodes),
    "gold_nodes": len(gold_nodes),
    "metric_nodes": len(metric_nodes),
    "is_dag": nx.is_directed_acyclic_graph(G),
    "timestamp": datetime.now().isoformat()
}

stats_file = f"{LOCAL_DATA_DIR}/geographic_equity_dag_statistics.json"
with open(stats_file, 'w') as f:
    json.dump(stats, f, indent=2)
print(f"  ✓ Saved statistics: {stats_file}")


SAVING DAG AND METADATA
  ✓ Saved DAG: ./3_data/geographic_health_equity_dag.graphml
  ✓ Saved RAG data: ./3_data/geographic_health_equity_rag_data.json
  ✓ Saved statistics: ./3_data/geographic_equity_dag_statistics.json


# STEP 7: Summary

In [38]:
print("\n" + "="*80)
print("GEOGRAPHIC HEALTH EQUITY PIPELINE COMPLETE")
print("="*80)

print(f"\nData Sources:")
print(f"  - OMOP Clinical: 8 tables")
print(f"  - Medicare: 2 tables")
print(f"  - Dual Enrollment SDOH: 1 table")

print(f"\nPipeline Architecture:")
print(f"  - Bronze Layer: {len(bronze_nodes)} tables (raw geographic & clinical data)")
print(f"  - Silver Layer: {len(silver_nodes)} tables (regional aggregations)")
print(f"  - Gold Layer: {len(gold_nodes)} tables (vertical transformations)")
print(f"  - Metrics Layer: {len(metric_nodes)} final KPIs")

print(f"\nFinal Metrics:")
print(f"  1. Health Equity Index by Region")
print(f"  2. Care Desert Score")
print(f"  3. Social Determinants Impact")
print(f"  4. Geographic Disparity Magnitude")

print(f"\nDAG:")
print(f"  - Total transformations: {G.number_of_nodes()} nodes")
print(f"  - Data dependencies: {G.number_of_edges()} edges")
print(f"  - Valid DAG: {nx.is_directed_acyclic_graph(G)}")

print(f"\nCost Optimization:")
print(f"  - Strategy: LIMIT {LIMIT} on all BigQuery reads")
print(f"  - Storage: Local CSV (Pandas -> Spark)")

print("\n" + "="*80)
print("✓ Ready for Marquez lineage tracking integration")
print("✓ Ready for regional health equity analysis")
print("="*80)


GEOGRAPHIC HEALTH EQUITY PIPELINE COMPLETE

Data Sources:
  - OMOP Clinical: 8 tables
  - Medicare: 2 tables
  - Dual Enrollment SDOH: 1 table

Pipeline Architecture:
  - Bronze Layer: 11 tables (raw geographic & clinical data)
  - Silver Layer: 6 tables (regional aggregations)
  - Gold Layer: 15 tables (vertical transformations)
  - Metrics Layer: 4 final KPIs

Final Metrics:
  1. Health Equity Index by Region
  2. Care Desert Score
  3. Social Determinants Impact
  4. Geographic Disparity Magnitude

DAG:
  - Total transformations: 36 nodes
  - Data dependencies: 54 edges
  - Valid DAG: True

Cost Optimization:
  - Strategy: LIMIT 1000 on all BigQuery reads
  - Storage: Local CSV (Pandas -> Spark)

✓ Ready for Marquez lineage tracking integration
✓ Ready for regional health equity analysis


25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 124


25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 124
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 124
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 124


In [39]:
# Verify Lineage
print("\n" + "="*60)
print("LINEAGE TRACKING COMPLETE!")
print("="*60)
print(f"✅ Notebook completed successfully")
print(f"✅ Marquez Web UI: http://localhost:3601")
print(f"✅ Namespace: geographic_equity")  # CORRECTED!
print(f"\nNext Steps:")
print("1. Open http://localhost:3601 in your browser")
print("2. Select namespace: 'geographic_equity'")
print("3. Browse Jobs and Datasets")
print("4. Click on any dataset to see lineage graph")
print("="*60)

# CRITICAL: Stop Spark to send job completion event
print("\nStopping Spark session to complete job...")
spark.stop()
print("✅ Spark session stopped - job marked as COMPLETE in Marquez")


LINEAGE TRACKING COMPLETE!
✅ Notebook completed successfully
✅ Marquez Web UI: http://localhost:3601
✅ Namespace: geographic_equity

Next Steps:
1. Open http://localhost:3601 in your browser
2. Select namespace: 'geographic_equity'
3. Browse Jobs and Datasets
4. Click on any dataset to see lineage graph

Stopping Spark session to complete job...


25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 125
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 125
25/11/23 20:06:45 ERROR ContextFactory: Query execution is null: can't emit event for executionId 125


✅ Spark session stopped - job marked as COMPLETE in Marquez
